# Cadrille — Google Colab

Supports:
- **Eval**: run inference + metrics on a checkpoint (free T4 / Pro A100)
- **SFT smoke test**: 50-step training run to verify the pipeline
- **RL fine-tuning**: Dr. CPPO on H100 (Pro+ recommended)
- **Monitoring**: sync W&B offline runs

> Mount Google Drive first — data and checkpoints survive session restarts there.

In [ ]:
# ── [1] Mount Drive ──────────────────────────────────────────────────────────
# All data and checkpoints live here — they survive session restarts.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/cadrille'   # adjust if needed
DRIVE_DATA  = f'{DRIVE_ROOT}/data'
DRIVE_CKPTS = f'{DRIVE_ROOT}/checkpoints'

import os
os.makedirs(DRIVE_DATA,  exist_ok=True)
os.makedirs(DRIVE_CKPTS, exist_ok=True)
print('Drive mounted. Paths:', DRIVE_DATA, DRIVE_CKPTS)

In [ ]:
# ── [2] Clone repo ───────────────────────────────────────────────────────────
import os

REPO = '/content/cadrille'
if not os.path.exists(REPO):
    !git clone https://github.com/miachen0401/cadrille.git {REPO}
else:
    !git -C {REPO} pull   # update to latest

%cd {REPO}

In [ ]:
# ── [3] Install dependencies ─────────────────────────────────────────────────
# flash-attn install strategy (in order, fastest to slowest):
#   1. GitHub prebuilt wheel  — exact match for this torch/cuda/python (~30 s)
#   2. PyPI                   — covers newer torch versions if wheel exists (~60 s)
#   3. Source build MAX_JOBS=8 — always works, ~10 min
# cadquery: cadquery-ocp MUST be installed before cadquery (avoids egg_info error)

import sys, subprocess, os
import torch

# ── flash-attn ────────────────────────────────────────────────────────────────
FA_VER = "2.7.2.post1"
_cuda  = torch.version.cuda.split('.')[0]            # "12" from "12.4"
_tmm   = '.'.join(torch.__version__.split('.')[:2])  # "2.5" from "2.5.1"
_py    = f"cp{sys.version_info.major}{sys.version_info.minor}"  # "cp311"
_whl   = f"flash_attn-{FA_VER}+cu{_cuda}torch{_tmm}-{_py}-{_py}-linux_x86_64.whl"
_url   = (f"https://github.com/Dao-AILab/flash-attention/releases/"
          f"download/v{FA_VER}/{_whl}")

print(f"flash-attn [1/3] GitHub prebuilt wheel: {_whl}")
_r = subprocess.run(["pip", "install", "-q", _url], capture_output=True, text=True)

if _r.returncode == 0:
    print("  → installed ✓")
else:
    # Wheel not on the pinned release (e.g. torch too new). Try PyPI — it carries
    # newer flash-attn releases with wheels for current torch versions.
    # Without --no-build-isolation, pip fails *fast* if no wheel available
    # (missing CUDA headers in isolated env), so this doesn't block long.
    print(f"  → not found (torch {_tmm} too new for {FA_VER}); trying PyPI ...")
    _r2 = subprocess.run(
        ["pip", "install", "-q", f"flash-attn>={FA_VER}"],
        capture_output=True, text=True)

    if _r2.returncode == 0:
        print("  → PyPI wheel installed ✓")
    else:
        print("  → no prebuilt wheel on PyPI; building from source (MAX_JOBS=8, ~10 min) ...")
        subprocess.run(
            ["pip", "install", "-q", "--no-build-isolation", f"flash-attn>={FA_VER}"],
            env={**os.environ, "MAX_JOBS": "8"}, check=True)
        print("  → built from source ✓")

# ── cadquery ──────────────────────────────────────────────────────────────────
# Install OCP bindings FIRST, then cadquery — reversing this order (or doing it
# in one pip call) causes "python setup.py egg_info did not run successfully".
# cadquery-ocp 7.7.2 has wheels for Python 3.9–3.11; 7.7.2.1 adds Python 3.12.
_ocp = "7.7.2.1" if sys.version_info >= (3, 12) else "7.7.2"
subprocess.run(["pip", "install", "-q", f"cadquery-ocp=={_ocp}"], check=True)
subprocess.run(["pip", "install", "-q", "cadquery==2.4.0"], check=True)
print(f"cadquery 2.4.0 + cadquery-ocp {_ocp} ✓")

# ── everything else ───────────────────────────────────────────────────────────
subprocess.run(["pip", "install", "-q",
    "transformers==4.50.3",
    "accelerate==0.34.2",
    "qwen-vl-utils==0.0.10",
    "bitsandbytes>=0.43.0",
    "trimesh==4.5.3",
    "open3d",
    "scipy==1.14.1",
    "wandb",
    "tqdm",
    "pyyaml",
], check=True)

# ── verify ────────────────────────────────────────────────────────────────────
p = torch.cuda.get_device_properties(0)
print(f"\nGPU : {p.name}  {p.total_memory // 1024**3} GB")
print(f"CUDA: {torch.version.cuda}  |  torch {torch.__version__}  |  Python {sys.version.split()[0]}")

In [ ]:
# ── [4] Link data & checkpoints from Drive ───────────────────────────────────
import os

if not os.path.exists('data'):
    os.symlink(DRIVE_DATA, 'data')

if not os.path.exists('checkpoints'):
    os.symlink(DRIVE_CKPTS, 'checkpoints')

print('data/      →', os.listdir('data') or '(empty)')
print('checkpoints/ →', os.listdir('checkpoints') or '(empty)')

In [ ]:
# ── [5] Download training & validation data (run once, persists on Drive) ──────
# Downloads DeepCAD test meshes (RL training + validation) and Fusion360 test
# meshes (validation only). Each is a separate HuggingFace dataset repo.
# Skip this cell if both folders already exist in data/.

import os

if not os.path.exists('data/deepcad_test_mesh'):
    # ~1 GB — DeepCAD test split (STL meshes); used as RL training prompts + eval
    !huggingface-cli download maksimko123/deepcad_test_mesh \
        --repo-type dataset \
        --local-dir data/deepcad_test_mesh

if not os.path.exists('data/fusion360_test_mesh'):
    # ~200 MB — Fusion360 test split; used for cross-dataset validation only
    !huggingface-cli download maksimko123/fusion360_test_mesh \
        --repo-type dataset \
        --local-dir data/fusion360_test_mesh

print('Data ready:')
!ls data/deepcad_test_mesh | head -5
!echo "  ... $(ls data/deepcad_test_mesh | wc -l) deepcad meshes"
!echo "  ... $(ls data/fusion360_test_mesh | wc -l) fusion360 meshes"

In [ ]:
# ── [6] Download SFT checkpoint (run once, persists on Drive) ─────────────────
# Starting point for RL fine-tuning.
# Skip if checkpoints/cadrille-sft already exists on Drive.

import os

SFT_CKPT = 'checkpoints/cadrille-sft'
if not os.path.exists(SFT_CKPT):
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='maksimko123/cadrille',
        local_dir=SFT_CKPT)
    print(f'Downloaded → {SFT_CKPT}')
else:
    print(f'Already exists: {SFT_CKPT}')

In [ ]:
# ── [7] W&B login ────────────────────────────────────────────────────────────
import wandb
wandb.login()   # paste your API key from wandb.ai/authorize

In [ ]:
# ── [8] RL fine-tuning ────────────────────────────────────────────────────────
# Auto-selects config based on GPU:
#   H100  80 GB → configs/rl/h100.yaml   G=16, batch_updates=3  (~8-12 steps/min)
#   A100  80 GB → configs/rl/h100.yaml   (same memory budget)
#   A100  40 GB → configs/rl/a100.yaml   G=8,  batch_updates=2  (~5-8  steps/min)
#   RTX 4080 16 GB → configs/rl/4080.yaml G=4, sequential       (~2-4  steps/min)
#
# Checkpoints saved to Drive every 500 steps; eval every 500 steps.
# W&B run ID saved to checkpoints/<run_name>/wandb_run_id.txt for resume.

import torch

_gpu_name = torch.cuda.get_device_name(0)
_vram_gb  = torch.cuda.get_device_properties(0).total_memory // 1024**3
print(f"Detected: {_gpu_name}  {_vram_gb} GB")

if _vram_gb >= 70:          # H100 80 GB or A100 80 GB
    CONFIG = 'configs/rl/h100.yaml'
elif _vram_gb >= 35:        # A100 40 GB
    CONFIG = 'configs/rl/a100.yaml'
else:                       # RTX 4080 / other 16 GB
    CONFIG = 'configs/rl/4080.yaml'

print(f"Using config: {CONFIG}")

RUN_NAME = 'cadrille-rl-v1'   # change for a new run; keep same to resume

!python rl/train.py \
    --config  {CONFIG} \
    --run-name {RUN_NAME}

In [ ]:
# ── [9] Resume after session timeout ─────────────────────────────────────────
# Colab sessions timeout (~12h Pro, ~24h Pro+).
# When your session restarts:
#   1. Re-run cells [1] – [7] (Drive mount, clone, install, link, login)
#   2. Run this cell to continue from the latest saved checkpoint.
#
# The step counter resumes at the checkpoint step (no duplicate steps logged).
# W&B attaches to the original run via the saved run ID in wandb_run_id.txt.

import glob, torch

RUN_NAME = 'cadrille-rl-v1'   # must match the original run

# Auto-select same config as the original run
_vram_gb = torch.cuda.get_device_properties(0).total_memory // 1024**3
if _vram_gb >= 70:
    CONFIG = 'configs/rl/h100.yaml'
elif _vram_gb >= 35:
    CONFIG = 'configs/rl/a100.yaml'
else:
    CONFIG = 'configs/rl/4080.yaml'

ckpt_dirs = sorted(glob.glob(f'checkpoints/{RUN_NAME}/checkpoint-[0-9]*'),
                   key=lambda p: int(p.rsplit('-', 1)[-1]))
if not ckpt_dirs:
    print('No checkpoints found — start a fresh run with cell [8] instead.')
else:
    latest = ckpt_dirs[-1]
    step   = int(latest.rsplit('-', 1)[-1])
    print(f'GPU: {torch.cuda.get_device_name(0)}  {_vram_gb} GB  →  {CONFIG}')
    print(f'Resuming from: {latest}  (step {step} → 50000)')
    !python rl/train.py \
        --config           {CONFIG} \
        --run-name         {RUN_NAME} \
        --checkpoint-path  {latest}